# FRED Time Series — MCP & TimeSeriesDataFetcherAgent Tests

Tests the MCP interface directly and via the `TimeSeriesDataFetcherAgent`.

In [1]:
import os
import sys
import json

sys.path.insert(0, os.path.abspath('../../..'))

from apps.agentic.core.utils import set_anthropic_env

set_anthropic_env(filedir='../../../.keys')

## 1. Direct MCP Tool Call

Verify the MCP server is reachable and `fred.series_observations` returns expected data.

In [2]:
from lib.mcp_client import MCPClient, MCPClientConfig
from apps.agentic.core.constants import MCP_URL

mcp_config = MCPClientConfig(url=MCP_URL)

async def call_tool(tool_name, arguments=None):
    async with MCPClient(mcp_config) as client:
        return await client.call_tool(tool_name, arguments or {})

async def list_mcp_tools():
    async with MCPClient(mcp_config) as client:
        return await client.list_tools()

tools = await list_mcp_tools()
for t in tools:
    print(f'{t.name}: {t.description}')

fred_category_children: List the child categories for a FRED category.
fred_category_series: List the series contained within a FRED category.
fred_series_info: Fetch metadata for a single FRED series.
fred_series_observations: Return observations for a series (limit defaults to 100).
fred_series_updates: Return recently updated FRED series.
list_releases: List FRED releases — the publications data arrives in, such as the Employment Situation or the H.15 selected interest rates. Returns the release ids that fred_release_series takes.
fred_release_series: List the series that belong to a FRED release.
tiingo_series_info: Fetch metadata for a Tiingo ticker (ETF, mutual fund, or stock), including its available date range.
tiingo_price_series: Return the end-of-day price series (OHLCV plus split/dividend-adjusted prices) for a Tiingo ticker. Provide start_date/end_date as YYYY-MM-DD for a range; omit for the latest day.
bls_series_data: Fetch observations for one or more BLS series IDs (up

In [4]:
SERIES_ID = 'BOGMBBM'

result = await call_tool('fred_series_observations', {'series_id': SERIES_ID, 'limit': 5})

print(result)
if result.structured_content:
    # mcp 2.x stopped wrapping the payload under a 'result' key, and the reply no longer
    # echoes `limit`/`offset` -- both are still accepted arguments, they are just not
    # returned. The schema takes series_id, limit, offset, frequency, units.
    data = result.structured_content
    print(f"series_id : {data['series_id']}")
    print(f"count     : {data['count']}  (total in the series, NOT the number returned)")
    print(f"returned  : {len(data['observations'])}")
    print()
    for obs in data['observations']:
        print(obs)

meta=None content=[TextContent(type='text', text='{\n  "series_id": "BOGMBBM",\n  "observations": [\n    {\n      "date": "1959-01-01",\n      "value": "18.9"\n    },\n    {\n      "date": "1959-02-01",\n      "value": "18.6"\n    },\n    {\n      "date": "1959-03-01",\n      "value": "18.4"\n    },\n    {\n      "date": "1959-04-01",\n      "value": "18.7"\n    },\n    {\n      "date": "1959-05-01",\n      "value": "18.6"\n    }\n  ],\n  "count": 813\n}', annotations=None, meta=None)] structured_content={'series_id': 'BOGMBBM', 'observations': [{'date': '1959-01-01', 'value': '18.9'}, {'date': '1959-02-01', 'value': '18.6'}, {'date': '1959-03-01', 'value': '18.4'}, {'date': '1959-04-01', 'value': '18.7'}, {'date': '1959-05-01', 'value': '18.6'}], 'count': 813} is_error=False result_type='complete'
series_id : BOGMBBM
count     : 813  (total in the series, NOT the number returned)
returned  : 5

{'date': '1959-01-01', 'value': '18.9'}
{'date': '1959-02-01', 'value': '18.6'}
{'date': '1

In [5]:
from datetime import datetime

result = await call_tool('fred_series_observations', {'series_id': SERIES_ID, 'limit': 10000})

if result.structured_content:
    observations = result.structured_content['observations']

    timestamps = []
    values = []
    for obs in observations:
        if obs['value'] == '.':
            continue
        timestamps.append(datetime.strptime(obs['date'], '%Y-%m-%d'))
        values.append(float(obs['value']))

    print(f'Series       : {SERIES_ID}')
    print(f'Observations : {len(observations)}')
    print(f'Numeric      : {len(timestamps)}  (missing values, "." , dropped)')
    print(f'Start        : {timestamps[0].date()}')
    print(f'End          : {timestamps[-1].date()}')
    print(f'Min val      : {min(values):.4f}')
    print(f'Max val      : {max(values):.4f}')

Series       : BOGMBBM
Observations : 813
Numeric      : 813  (missing values, "." , dropped)
Start        : 1959-01-01
End          : 2026-09-01
Min val      : 12.3000
Max val      : 4193.2000


## 2. MCP Tool Discovery via `MCPAdapter`

Verify that langchain's in-tree MCP bridge discovers tools from the server. This is the
layer `MCPToolRegistry` sits on, so a failure here localises the fault below the registry.

`langchain-mcp-adapters` used to do this job and is gone: it pins `mcp<2` and the repo is
on `mcp==2.3.0`. The `[mcp]` extra on `langchain` replaces it, and `get_tools()` is now
`list_tools()`.

In [6]:
from typing import Any
from langchain.mcp import MCPAdapter

connections: dict[str, Any] = {'fred': {'transport': 'sse', 'url': MCP_URL}}
discovered_tools = await MCPAdapter(connections).list_tools()

print(f'Discovered {len(discovered_tools)} tools:')
for t in discovered_tools:
    print(f'  {t.name}: {t.description}')

Discovered 32 tools:
  fred_category_children: List the child categories for a FRED category.
  fred_category_series: List the series contained within a FRED category.
  fred_series_info: Fetch metadata for a single FRED series.
  fred_series_observations: Return observations for a series (limit defaults to 100).
  fred_series_updates: Return recently updated FRED series.
  list_releases: List FRED releases — the publications data arrives in, such as the Employment Situation or the H.15 selected interest rates. Returns the release ids that fred_release_series takes.
  fred_release_series: List the series that belong to a FRED release.
  tiingo_series_info: Fetch metadata for a Tiingo ticker (ETF, mutual fund, or stock), including its available date range.
  tiingo_price_series: Return the end-of-day price series (OHLCV plus split/dividend-adjusted prices) for a Tiingo ticker. Provide start_date/end_date as YYYY-MM-DD for a range; omit for the latest day.
  bls_series_data: Fetch observ

/var/folders/3m/ycv0bg1d0_37hws7nxp1rhgr0000gn/T/ipykernel_67921/2510474784.py:2: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


In [7]:
from apps.agentic.core.mcp_tool_registry import MCPToolRegistry
from apps.agentic.core.constants import MCP_URL

await MCPToolRegistry.initialize(
    servers={"fred": {"transport": "sse", "url": MCP_URL}},
    required=["fred_series_observations"],
)

DEBUG:    MCPToolRegistry discovered 32 tools: ['fred_category_children', 'fred_category_series', 'fred_series_info', 'fred_series_observations', 'fred_series_updates', 'list_releases', 'fred_release_series', 'tiingo_series_info', 'tiingo_price_series', 'bls_series_data', 'bls_series_latest', 'bls_popular_series', 'bls_all_surveys', 'bls_survey_info', 'bis_dataflows', 'bis_datastructure', 'bis_series_data', 'fbi_offenses', 'fbi_arrests', 'fbi_employment', 'cdc_discover', 'cdc_categories', 'cdc_tags', 'cdc_dataset_columns', 'cdc_series_data', 'cdc_dataset_facets', 'timeseries_source_list', 'timeseries_source_data', 'timeseries_source_stale', 'series_catalog_search', 'series_catalog_entry', 'series_catalog_concepts']
INFO:     MCPToolRegistry initialized with 32 tools: ['fred_category_children', 'fred_category_series', 'fred_series_info', 'fred_series_observations', 'fred_series_updates', 'list_releases', 'fred_release_series', 'tiingo_series_info', 'tiingo_price_series', 'bls_series_dat

## 3. TimeSeriesDataFetcherAgent — Direct Invocation

Create the agent via `TimeSeriesDataFetcherAgent.create()` and invoke it directly. The agent wraps the MCP tools with caching, so it stores the full series in `SeriesCache` and returns a compact `SeriesRef` JSON (not the raw observations).

In [8]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
import shortuuid

from apps.agentic.db.series_cache import SeriesCache
from apps.agentic.agents.time_series.time_series_data_fetcher_agent import TimeSeriesDataFetcherAgent

SeriesCache.initialize()

agent = await TimeSeriesDataFetcherAgent.create()
print(f'Agent initialized with tools: {list(agent.tool_node.tools_by_name.keys())}')

INFO:     SeriesCache initialized: postgresql+psycopg2://yada@localhost/yada_dev
DEBUG:    TimeSeriesDataFetcherAgent using MCP tools: ['fred_series_observations', 'fred_series_info', 'tiingo_price_series', 'tiingo_series_info']
DEBUG:    TimeSeriesDataFetcherAgent prompt: 
            <instructions>
            You are a time series data fetcher agent. Use the available MCP tools to retrieve time series
            data from external data sources.

            The tools return a SeriesRef JSON object — pass it back unmodified as your response.
            Do not summarize, reformat, or interpret it.

            When fetching FRED data use the fred_series_observations tool with a limit of
            10000 to retrieve the full series.

            For ETF, mutual fund, or stock price history, use the tiingo_price_series tool
            with the ticker symbol (e.g. SPY); it returns the full adjusted-close price series.
            </instructions>
            
Agent initialized with to

In [11]:
from apps.agentic.core.agents.messages import WorkerState

# Annotated, not bare: the graph's input type is the WorkerState TypedDict. A dict
# literal only checks against a TypedDict through an annotation -- an inferred
# dict[str, list[HumanMessage]] is not assignable, which is the Pylance
# reportArgumentType. Matches plot_agent.py:49. Runtime behaviour is unchanged.
state: WorkerState = {
    'messages': [HumanMessage(content=f'Fetch the {SERIES_ID} series from FRED with a limit of 10000.')]
}
run_config = RunnableConfig(configurable={'thread_id': shortuuid.uuid()})

result = await agent.agent.ainvoke(state, run_config)
response = result['messages'][-1].content
print('Agent response (SeriesRef):', response)

INFO:     [TimeSeriesDataFetcherAgent] routing → fred_series_observations({'series_id': 'BOGMBBM', 'limit': 10000})
DEBUG:    fred_series_observations: cache hit for fred:BOGMBBM
Agent response (SeriesRef): {"source": "fred", "native_id": "BOGMBBM", "cache_id": "55c25b87-5cb7-4aba-8163-9e933ad5ef04"}
